#### LnagGraph

- 워크플로 프레임워크
- LCEL 선형(A -> B -> C) / LangGraph 순환(A -> B -> A -> C -> B...) 흐름 지원
- 개념
    - Node : 실행할 함수
    - Edge : 노드 간 연결
    - State : 전체 상태를 담는 dict
- 조건부 Edge, Agent Loop, 자기수정 Multi Agent 등 복잡한 패턴 구현

LangChain | LangGraph 중 선택하면된다

In [1]:
!pip3 install langgraph grandalf

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

#### TypedDict / Pydantic
- TypedDic : 딕셔너리 키, 값 타입을 정의할 수 있게 도와줌, 단순 State 구현시 사용, 기본값 줄 수 없음
- Pydantic : validation 검사(입력값), 검증이 필요한 State인 경우 사용

In [2]:
# 데이터 저장소 생성
# message 상태관리 => 공유
class MyState(TypedDict):
    message:str

# 작업 함수 생성(노드)
def say_hello(state):
    # state 값의 변화
    return{'message':"Hello, LangGraph"}

# 그래프 생성
graph = StateGraph(MyState)
graph.add_node("hello",say_hello)

graph.add_edge(START,"hello")
graph.add_edge("hello", END)

# 실행
app = graph.compile()
result = app.invoke({"message":""})
print(result)

# 그래프 시각화(참고)
app.get_graph().print_ascii()

{'message': 'Hello, LangGraph'}
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | hello |    
  +-------+    
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


#### State
- 상태를 중심으로 동작
- TypedDict, Pydantic Base Model 을 사용하여 정의
- 그래프 실행  중 지속적으로 업데이트 됨
- 노드간의 전환은 조건부 엣지를 통해 제어 가능하며 복잡한 의사결정 프로세스 모델링 가능
- 재귀적 실행 지원

#### Node
- 실제 작업을 수행하는 기본 단위
- 함수 기반
- 상태 중심 : 현제 상태를 입력으로 받아 처리
- 독립적 실행 : 각 노드는 독립적으로 실행
- 조합 가능 : 여러 노드 연결하여 복잡한 워크 플로우 가능

#### Edge
- 노드간의 연결과 실행 흐름을 정의

In [12]:
# 카운터 공유

class CounterState(TypedDict):
    count:int

# 증가 함수
def increment(state):
    print(f"현재 카운트 : {state['count']}")
    new_count = state['count'] = 1
    print(f"새로운 카운트 : {new_count}")

    # state 값 변경(return)
    return {"count":new_count}

# 그래프 생성
graph = StateGraph(CounterState)
graph.add_node("increment",increment)

# 그래프 연결(그래프가 실행 될것인가?)
graph.add_edge(START,"increment")
graph.add_edge("increment",END)

# 그래프를 실행 가능한 형태로 변경
app = graph.compile()
result = app.invoke({"count":0})
print(f"최종 결과 {result}")

# 그래프 시각화(참고)
app.get_graph().print_ascii()

현재 카운트 : 0
새로운 카운트 : 1
최종 결과 {'count': 1}
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+-----------+  
| increment |  
+-----------+  
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [13]:
# 2개의 노드

def first_increment(state):
    # +1
    return {"count":state["count"]+1}
def second_increment(state):
    # +10
    return {"count":state["count"]+10}

# START => first => second => END
graph = StateGraph(CounterState)
graph.add_node("first",first_increment)
graph.add_node("second",second_increment)

# 그래프 연결(그래프가 실행 될것인가?)
graph.add_edge(START,"first")
graph.add_edge("first","second")
graph.add_edge("second",END)

# 그래프를 실행 가능한 형태로 변경
app = graph.compile()
result = app.invoke({"count":0})
print(f"최종 결과 {result}")

# 그래프 시각화(참고)
app.get_graph().print_ascii()

최종 결과 {'count': 11}
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +-------+    
  | first |    
  +-------+    
      *        
      *        
      *        
  +--------+   
  | second |   
  +--------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


- 조건부 Edge
    - RunTime 상태에 따라 동적으로 실행 경로 결정
    - add_conditional_edges()

In [19]:
# input_num > 10 => big
# input_num < 10 => small

# State : number, result
class NumberState(TypedDict):
    number:int
    result:str

# Node(result값 변경)
def handle_big_number(state):
    return {'result':f"{state['number']}는 큰 숫자입니다."} 

def handle_small_number(state):
    return {'result':f"{state['number']}는 작은 숫자입니다."} 

# routor(조건함수)
def check_size(state):
    if state['number'] > 10:
        return "big"
    else:
        return "small"
# Graph생성
graph = StateGraph(NumberState)
graph.add_node("big_handler",handle_big_number)
graph.add_node("small_handler",handle_small_number)

# Edge
graph.add_edge("big_handler",END)
graph.add_edge("small_handler",END)

# 조건부 Edge(number 값에 따라 big or small)
graph.add_conditional_edges(START, check_size,{"big":"big_handler","small":"small_handler"})


app = graph.compile()
result=app.invoke({"number":15, "result":""})
print(f"큰 숫자 {result}")
result=app.invoke({"number":5, "result":""})
print(f"작은 숫자 {result}")

# 그래프 시각화(참고)
app.get_graph().print_ascii()

큰 숫자 {'number': 15, 'result': '15는 큰 숫자입니다.'}
작은 숫자 {'number': 5, 'result': '5는 작은 숫자입니다.'}
              +-----------+                 
              | __start__ |                 
              +-----------+                 
              ..           ..               
            ..               ..             
          ..                   ..           
+-------------+           +---------------+ 
| big_handler |           | small_handler | 
+-------------+           +---------------+ 
              **           **               
                **       **                 
                  **   **                   
                +---------+                 
                | __end__ |                 
                +---------+                 


In [ ]:
# 홀, 짝
# State : number, result
class NumberState(TypedDict):
    number:int
    result:str

# Node(result값 변경)
def even_node(state):
    return {'result':f"{state['number']}는 짝수입니다."} 

def odd_node(state):
    return {'result':f"{state['number']}는 홀수입니다."} 

# routor(조건함수)
def check_number(state):
    if state['number'] % 2 == 0:
        return "even"
    else:
        return "odd"
# Graph생성
graph = StateGraph(NumberState)
graph.add_node("even_node",even_node)
graph.add_node("odd_node",odd_node)

# Edge
graph.add_edge("even_node",END)
graph.add_edge("odd_node",END)

# 조건부 Edge(number 값에 따라 even or odd)
graph.add_conditional_edges(START, check_number,{"even":"even_node","odd":"odd_node"})


app = graph.compile()
result=app.invoke({"number":14, "result":""})
print(f"짝수 {result}")
result=app.invoke({"number":5, "result":""})
print(f"홀수 {result}")

# 그래프 시각화(참고)
app.get_graph().print_ascii()

짝수 {'number': 14, 'result': '14는 짝수입니다.'}
홀수 {'number': 5, 'result': '5는 홀수입니다.'}
            +-----------+             
            | __start__ |             
            +-----------+             
           ...         ...            
          .               .           
        ..                 ..         
+-----------+           +----------+  
| even_node |           | odd_node |  
+-----------+           +----------+  
           ***         ***            
              *       *               
               **   **                
             +---------+              
             | __end__ |              
             +---------+              


In [26]:
# 학점
class ScoreState(TypedDict):
    score : int

# Node return => 상태값 변경
# return 안하면 None
def grade_a(state):
    print("A학점")
    # score = None
    return {}

def grade_b(state):
    print("B학점")
    return {}

def grade_c(state):
    print("C학점")
    return {}

def route_grade(state):
    if state['score'] >= 90:
        return "A"
    elif state['score'] >= 80:
        return "B"
    else:
        return "C"

# score >= 90 : A, score >= 80 : B, C

graph = StateGraph(ScoreState)
graph.add_node("grade_a", grade_a) 
graph.add_node("grade_b", grade_b) 
graph.add_node("grade_c", grade_c) 

graph.add_edge("grade_a",END)
graph.add_edge("grade_b",END)
graph.add_edge("grade_c",END)

graph.add_conditional_edges(START, route_grade,{"A":"grade_a","B":"grade_b","C":"grade_c"})

app = graph.compile()
result=app.invoke({"score":93})
print(f"점수 {result}")
result=app.invoke({"score":75})
print(f"점수 {result}")
result=app.invoke({"score":85})
print(f"점수 {result}")

# 그래프 시각화(참고)
app.get_graph().print_ascii()


A학점
점수 {'score': 93}
C학점
점수 {'score': 75}
B학점
점수 {'score': 85}
                     +-----------+                       
                     | __start__ |                       
                   ..+-----------+...                    
               ....         .        ....                
           ....             .            ....            
         ..                 .                ..          
+---------+           +---------+           +---------+  
| grade_a |           | grade_b |           | grade_c |  
+---------+****       +---------+        ***+---------+  
               ****         *        ****                
                   ****     *    ****                    
                       **   *  **                        
                      +---------+                        
                      | __end__ |                        
                      +---------+                        


In [33]:
# 상태관리 :text, sentiment, result
# text : 오늘 기분이 너무 좋아 
# analyze_Sentiment() : 감정평가 text 좋다 positive/싫다 negative/other neutral
# positive_node() : 긍정 의견 | negative_node() : 부정 의견 | neutral_node() : 중립
# route_sentiment() : return state['snetiment']

# START => analyze => positive
#                  => negative
#                  => neutral

class SentimentState(TypedDict):
    text : str
    sentiment : str
    result : str

def analyze_sentiment_node(state: SentimentState):
    text = state["text"]
    # 간단한 키워드 기반 매핑 (실제로는 여기에 LLM이나 AI 모델이 들어갑니다)
    if "좋" in text:
        sentiment = "positive"
    elif "싫" in text:
        sentiment = "negative"
    else:
        sentiment = "neutral"
        
    return {"sentiment": sentiment} # 감정 분석 결과를 State에 저장!

def analyze_positive(state):
    print("positive")
    # score = None
    return {}

def analyze_negative(state):
    print("negative")
    return {}

def analyze_neutral(state):
    print("neutral")
    return {}


def route_sentiment(state: SentimentState):
    return state["sentiment"]

graph = StateGraph(SentimentState)

# 노드 등록 (감정분석 노드 추가)
graph.add_node("analyzer", analyze_sentiment_node)
graph.add_node("sentiment_pos", analyze_positive) 
graph.add_node("sentiment_neg", analyze_negative) 
graph.add_node("sentiment_neu", analyze_neutral) 

# 일반 엣지 연결 (START는 무조건 먼저 analyzer 노드로 갑니다)
graph.add_edge(START, "analyzer")

# 엔드포인트 연결
graph.add_edge("sentiment_pos", END)
graph.add_edge("sentiment_neg", END)
graph.add_edge("sentiment_neu", END)

# 조건부 엣지 연결 (analyzer 노드가 끝난 '이후'에 감정에 따라 분기합니다)
graph.add_conditional_edges(
    "analyzer", 
    route_sentiment,
    {
        "positive": "sentiment_pos",
        "negative": "sentiment_neg",
        "neutral": "sentiment_neu"
    }
)

app = graph.compile()
result=app.invoke({"text":"기분이 너무 좋다"})
print(f"감정 평가 문구 {result}")
result=app.invoke({"text":"게으른 게 싫다"})
print(f"감정 평가 문구 {result}")
result=app.invoke({"text":"오늘도 활기찬 아침이군"})
print(f"감정 평가 문구 {result}")

# 그래프 시각화(참고)
app.get_graph().print_ascii()


positive
감정 평가 문구 {'text': '기분이 너무 좋다', 'sentiment': 'positive'}
negative
감정 평가 문구 {'text': '게으른 게 싫다', 'sentiment': 'negative'}
neutral
감정 평가 문구 {'text': '오늘도 활기찬 아침이군', 'sentiment': 'neutral'}
                              +-----------+                                
                              | __start__ |                                
                              +-----------+                                
                                    *                                      
                                    *                                      
                                    *                                      
                              +----------+                                 
                              | analyzer |..                               
                         .....+----------+  .....                          
                     ....           .            ....                      
                .....               .        

In [2]:
from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import ChatPromptTemplate

from langchain_ibm import WatsonxEmbeddings
from langchain_ibm import ChatWatsonx

from langchain_openai import ChatOpenAI

from langchain_ollama import ChatOllama

from dotenv import load_dotenv

import os

c:\souce\ollama\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
#.env 내용 가죠오기
load_dotenv()

apikey = os.getenv("WATSONX_API_KEY")
project_id = os.getenv("WATSONX_PROJECT_ID")
watsonx_ai_url = os.getenv("WATSONX_URL")
hf_token = os.getenv("HF_TOKEN")
cohere_api_key = os.getenv("COHERE_API_KEY")
serper_api_key = os.getenv("SERPER_API_KEY")

In [6]:
hugging_llm = ChatOpenAI(
    model="Qwen/Qwen2.5-7B-Instruct:together",
    api_key=hf_token,
    base_url="https://router.huggingface.co/v1",
    temperature= 0
)
# 유료 LLM 선언

watson_llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}",
    params = {
    "max_tokens": 2000,
    "temperature": 0
    }
)

# 로컬 LLM 선언
qwen_llm = ChatOllama(model="qwen3.5:4b",temperature= 0)

exaone_llm = ChatOllama(model="exaone3.5:2.4b",temperature= 0)

In [7]:
watsonx_enbedding = WatsonxEmbeddings(
    model_id="ibm/granite-embedding-278m-multilingual",
    url = f"{watsonx_ai_url}",
    api_key = f"{apikey}",
    project_id=f"{project_id}"
)

In [4]:
parsor = StrOutputParser()

translate_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트를 한국어로 번역하세요. 번역문만 출력\n{text}")
]) | watson_llm | parsor
summarize_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트를 3문장으로 요약하세요.\n{text}")
]) | watson_llm | parsor
sentiment_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트의 감정을 긍정/부정/중립 중 하나로만 답하세요.\n{summary}")
]) | watson_llm | parsor

In [ ]:
# State 
class AnalysisState(TypedDict):
    text:str
    translated:str
    summary:str
    sentiment:str
    done:bool

# Node 정의
def translated_node(state):
    result = translate_chain.invoke({"text":state['text']})
    return {"translated":result}

def summary_node(state):
    result = summarize_chain.invoke({"text":state['translated']})
    return {"summary":result}

def sentiment_node(state):
    result = sentiment_chain.invoke({"summary":state['summary']})
    return {"sentiment":result}

# 그래프 생성
graph = StateGraph(AnalysisState)
graph.add_node("translated",translated_node)
graph.add_node("summary",summary_node)
graph.add_node("sentiment",sentiment_node)

graph.add_edge(START,"translated")
graph.add_edge("translated","summary")
graph.add_edge("summary","sentiment")
graph.add_edge("sentiment",END)


app= graph.compile()
result = app.invoke({"text":"Python is great!","done":False})

print("번역 ",result['translated'])
print("요약 ",result['summary'])
print("감정 ",result['sentiment'])

# 그래프 시각화(참고)
app.get_graph().print_ascii()

번역  파이썬은 정말 멋져요!
요약  파이썬은 정말 멋진 프로그래밍 언어입니다. 파이썬은 간결하고 읽기 쉬운 문법을 가지고 있어 초보자에게도 적합합니다. 또한 다양한 라이브러리와 프레임워크를 제공하여 다양한 분야에서 활용할 수 있습니다.
감정  긍정
+-----------+  
| __start__ |  
+-----------+  
       *       
       *       
       *       
+------------+ 
| translated | 
+------------+ 
       *       
       *       
       *       
  +---------+  
  | summary |  
  +---------+  
       *       
       *       
       *       
+-----------+  
| sentiment |  
+-----------+  
       *       
       *       
       *       
  +---------+  
  | __end__ |  
  +---------+  


In [51]:
# 조건부 Edge : summay >= 100 자 이상 Good | 100자 미만 poor 

class SummaryStaste(TypedDict):
    text:str
    summary:str
    quality:str
    retries:int

def summary_node(state):
    result = summarize_chain.invoke({"text":state['text']})
    return {"summary":result}

# quality
def check_quality_node(state):
    # summay >= 100 자 이상 Good | 100자 미만 poor 
    quality = "good" if len(state['summary']) >= 100 else "poor"
    return {"quality":quality}

def retry_node(state):
    return {"text":state['text']+"\n 더 자세히 요약하시오.","retries":state['retries']+1}

def route_by_quality(state):
    if state['quality'] == "poor" and state['retries'] <3:
        return "retry"
    return "done"

graph = StateGraph(SummaryStaste)
graph.add_node("summary",summary_node)
graph.add_node("retry", retry_node)
graph.add_node("check_quality",check_quality_node)

# START => summary check_quality => 조건부 Edge
graph.add_edge(START,"summary")
graph.add_edge("summary","check_quality")
graph.add_conditional_edges("check_quality",route_by_quality,{
    "retry":"retry",
    "done":END
})
graph.add_edge("retry","summary")

app= graph.compile()
result = app.invoke({"text": "짧은 글.", "retries": 0})
print(f"최종 요약({len(result['summary'])}자):{result['summary'][:100]}")

app.get_graph().print_ascii()

최종 요약(249자):물론, 도와드리겠습니다! 다음은 텍스트의 3문장 요약입니다:

1. 텍스트는 다양한 상황에서 효과적인 의사소통을 위한 전략을 제공하는 자기계발서입니다.
2. 저자는 자신의 생각을 
              +-----------+          
              | __start__ |          
              +-----------+          
                    *                
                    *                
                    *                
              +---------+            
              | summary |            
              +---------+            
             ***         ***         
            *               *        
          **                 ***     
+---------------+               *    
| check_quality |               *    
+---------------+.              *    
        .         .....         *    
        .              ...      *    
        .                 ...   *    
   +---------+             +-------+ 
   | __end__ |             | retry | 
   +---------+             +-------+ 


In [53]:
class VerifyState(TypedDict):
    question:str
    answer:str
    feedback:str
    is_verified:bool
    attempt:int

def generate(state):
    """답변을 생성합니다. 이전 피드백이 있으면 반영합니다."""
    prompt = f"질문 : {state['question']}"

    if state['feedback']:
        prompt += f"\n이전 답변의 피드백 : {state['feedback']}\n위 피드백을 반영하여 개선된 답변을 작성하세요"

    result = watson_llm.invoke(prompt)
    return {'answer': result.content, "attempt":state['attempt']+1}
def verify(state):
    """답변의 정확성과 완전성을 검증합니다."""
    verification = watson_llm.invoke(f"""
다음 답변의 정확성과 완전성을 검증하세요
                      
질문:
{state['question']}

답변:
{state['answer']}

정확하고 완전하면 첫줄에 'PASS'를, 수정이 필요하면 첫줄에 'FAIL'을 쓰고 구체적인 개선 사항을 설명하세요. 
""")
    content = verification.content
    is_pass = content.strip().startswith('PASS')
    return {"is_verified":is_pass, "feedback":content}

# router
def should_retry(state):
    """검증 통과 또는 최대 횟수 도달 시 종료."""
    if state['is_verified'] or state['attempt'] >=3 :
        return END
    return "generate"


# 질문 => LLM 답변 생성 => LLM 답변 검증 => 검증 통과 
#                                     => 검증 미통과 => LLM 답변 생성 (Loop)
graph = StateGraph(VerifyState)
graph.add_node("generate",generate)
graph.add_node("verify",verify)

graph.add_edge(START,"generate")
graph.add_edge("generate","verify")

graph.add_conditional_edges("verify",should_retry,{"generate":"generate",END:END})

app= graph.compile()
result = app.invoke({
    "question":"파이썬에서 GIL이 무엇이며 멀티 쓰레딩에 어떤 영향을 주는지 설명해줘",
    "answer":"",
    "feedback":"",
    "is_verified":False,
    "attempt":0,
})

print("시도횟수",result['attempt'])
print("검증 통과",result['is_verified'])
print("최종 답변",result['answer'][:300])

시도횟수 1
검증 통과 True
최종 답변 GIL(Global Interpreter Lock)은 Python 인터프리터에서 사용되는 뮤텍스(Mutex)입니다. 이는 Python 코드가 동시에 실행되는 것을 방지하고, 한 번에 하나의 스레드만이 Python 바이트 코드를 실행할 수 있도록 합니다. GIL은 멀티 쓰레딩에 다음과 같은 영향을 미칩니다:

1. 멀티 쓰레딩의 효율성 저하: GIL로 인해 Python의 멀티 쓰레딩은 CPU 바운드 작업에 대해 효율적이지 않습니다. 이는 GIL이 한 번에 하나의 스레드만 실행할 수 있도록 하기 때문입니다. 따라서 멀티 쓰레딩을 사용하


In [8]:
from langchain_core.runnables import RunnableParallel
import time

In [55]:
class AnalysisState(TypedDict):
    text:str
    translated:str
    summary:str
    keywords:list[str]

translate_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트를 한국어로 번역하세요. 번역문만 출력\n{text}")
]) | watson_llm | parsor

summarize_chain = ChatPromptTemplate.from_messages([
    ("system",":다음 텍스트를 3문장으로 요약하세요.\n{text}")
]) | watson_llm | parsor

keyword_chain = ChatPromptTemplate.from_messages([
    ("system","키워드 5개 추출.\n{text}")
]) | watson_llm | parsor

def translate_node(state):
    print("번역 시작")
    time.sleep(3)
    print("번역 종료")
    return {"translated":translate_chain.invoke({"text":state['text']})}

def summarize_node(state):
    print("요약 시작")
    time.sleep(3)
    print("요약 종료")
    return {"summary":summarize_chain.invoke({"text":state['text']})}

def keyword_node(state):
    print("키워드 추출 시작")
    time.sleep(3)
    print("키워드 추출 종료")
    return {"keywords":keyword_chain.invoke({"text":state['text']})}

graph =StateGraph(AnalysisState)

graph.add_node('translated',translate_node)
graph.add_node('summary',summarize_node)
graph.add_node('keywords',keyword_node)

graph.add_edge(START,"translated")
graph.add_edge(START,"summary")
graph.add_edge(START,"keywords")

graph.add_edge("translated", END)
graph.add_edge("summary",END)
graph.add_edge("keywords",END)

app = graph.compile()
text = {"text": "Python is a versatile language used in Ai and web development"}

result = app.invoke(text)

print("번역",result['translated'][:100])
print("요약",result['summary'][:100])
print("키워드",result['keywords'][:100])

app.get_graph().print_ascii()

키워드 추출 시작
요약 시작
번역 시작
키워드 추출 종료
번역 종료
요약 종료
번역 파이썬은 AI와 웹 개발에 사용되는 다용도 언어입니다.
요약 파이썬은 인공지능과 웹 개발에 사용되는 다용도 언어입니다.
키워드 1. Python
2. versatile
3. language
4. Ai
5. web development
                      +-----------+                         
                      | __start__ |                         
                    **+-----------+****                     
                ****         *         ****                 
            ****             *             ****             
          **                 *                 **           
+----------+           +---------+           +------------+ 
| keywords |           | summary |           | translated | 
+----------+****       +---------+         **+------------+ 
                ****         *         ****                 
                    ****     *     ****                     
                        **   *   **                         
                        +---------+                         
                        | __e

- rag LangGraph
    - 사용자 질문 => retrieve => generate => END

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from typing import List
from langchain_core.documents import Document

C:\Users\soldesk\AppData\Local\Temp\ipykernel_10980\1217631854.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [10]:
# STEP 1 : 문서 로드
loader =PyPDFLoader("./data/Summary of ChatGPTGPT-4 Research.pdf")
# STEP 2 : 문서 분할
splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
chunks = splitter.split_documents(loader.load())
print(f"chunk 수 {chunks}")
# # STEP 3 : 인덱싱 - 임베딩
# embeddings = watsonx_enbedding(model="nomic-embed-text-v2-moe") 
# STEP 4 : 벡터스토어(Chroma or FAISS)
vectorstore = Chroma.from_documents(chunks, embedding=watsonx_enbedding,persist_directory="./db/chroma_db",collection_name="research")
# STEP 5 : as_retriever() : Vector Store => Retriever | connect LangChain
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":3})

chunk 수 [Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-04-05T00:33:07+00:00', 'author': '', 'keywords': '', 'moddate': '2023-04-05T00:33:07+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/Summary of ChatGPTGPT-4 Research.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Summary of ChatGPT/GPT-4 Research\nand Perspective Towards the Future of Large\nLanguage Models\nYiheng Liu ∗1, Tianle Han ∗1, Siyuan Ma 1, Jiayue Zhang 1,\nYuanyuan Yang1, Jiaming Tian 1, Hao He 1, Antong Li 2, Mengshen\nHe1, Zhengliang Liu 3, Zihao Wu 3, Dajiang Zhu 4, Xiang Li 5, Ning\nQiang1, Dingang Shen 6,7,8, Tianming Liu 3, and Bao Ge †1\n1School of Physics and Information Technology, Shaanxi Normal University, Xi’an\n710119 China'), Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX wi

In [65]:
class RAGState(TypedDict):
    query:str
    retrieved_docs:List[Document]
    answer:str

def retrieve(state):
    # 기존 벡터스토어 질의
    vectorstore = Chroma(collection_name="research", embedding_function=watsonx_enbedding,persist_directory="./db/chroma_db")

    docs = vectorstore.similarity_search(state['query'], k=3)

    return {"retrieved_docs":docs}

def generate(state):

    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    
    prompt = """\
다음 컨텍스트를 참고하여 질문에 답하세요.
컨텍스트에 없는 내용은 모른다고 답하세요.

컨텍스트:
{context}
질문:
{query}
"""

    response = watson_llm.invoke(prompt.format(context=context,query=state["query"]))
    return {"answer":response.content}

graph = StateGraph(RAGState)
graph.add_node("retrieve",retrieve)
graph.add_node("generate",generate)

graph.add_edge(START,"retrieve")
graph.add_edge("retrieve","generate")
graph.add_edge("generate",END)

app = graph.compile()
result = app.invoke({"query":"where can i use chatGPT?"})
print(result['answer'])

app.get_graph().print_ascii()

제공된 컨텍스트에 따르면, ChatGPT는 다음과 같은 분야에서 사용될 수 있습니다:

1. 교육 분야: 학생들은 ChatGPT를 사용하여 다양한 학문 분야(예: 물리학, 수학, 화학 등)에서 질문에 대한 답변을 찾고, 비교하며, 검증할 수 있습니다.

2. 코드 생성 분야: ChatGPT는 코드 생성에 사용될 수 있지만, 현재로서는 훈련 데이터가 Python, C++, Java와 같은 프로그래밍 언어에 편중되어 있어 다른 프로그래밍 언어나 코딩 스타일에 적용하기에는 한계가 있습니다.

컨텍스트에는 이 외에도 ChatGPT의 응용 분야에 대한 구체적인 언급이 없으므로, 위에서 언급한 분야 외에 다른 분야에서의 사용에 대해서는 알려진 정보가 없습니다.
+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| retrieve |   
+----------+   
      *        
      *        
      *        
+----------+   
| generate |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


In [14]:
# 기본 RAG + 평가

# 기존 벡터스토어 질의
vectorstore = Chroma(collection_name="research", embedding_function=watsonx_enbedding,persist_directory="./db/chroma_db")

class RAGState(TypedDict):
    query:str
    retrieved_docs:List[Document]
    answer:str
    is_relevant:bool
    retry_count:int

def retrieve(state):
    docs = vectorstore.similarity_search(state['query'], k=3)

    return {"retrieved_docs":docs}

def generate(state):

    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    
    prompt = """\
다음 컨텍스트를 참고하여 질문에 답하세요.
컨텍스트에 없는 내용은 모른다고 답하세요.

컨텍스트:
{context}
질문:
{query}
"""

    response = watson_llm.invoke(prompt.format(context=context,query=state["query"]))
    return {"answer":response.content}

def evaluate(state):
    """답변이 질문과 관련이 있는지 평가"""
    prompt = ChatPromptTemplate.from_template(
        """
질문:
{query}

답변:
{answer}

이답변이 질문에 적절히 대답하고 있나요? 'yes' 또는 'no'로만 답하세요.
"""
    )
    response = watson_llm.invoke(prompt.format(query=state['query'],answer=state['answer']))
    is_relevant = 'yes' in response.content.lower()
    return {"is_relevant":is_relevant,'retry_count':state['retry_count']+1}

# route
def should_retry(state):
    """재검사 여부 결정"""
    if state['is_relevant'] or state['retry_count'] >=2:
        return "done"
    return "retry"

# 그래프 구성
graph = StateGraph(RAGState)
graph.add_node("retrieve",retrieve)
graph.add_node("generate",generate)
graph.add_node("evaluate",evaluate)

graph.add_edge(START,"retrieve")
graph.add_edge("retrieve","generate")
graph.add_edge("generate","evaluate")

graph.add_conditional_edges("evaluate",should_retry,{"retry": "retrieve","done":END})

app = graph.compile()
result = app.invoke({
    "query":"where can i use chatGPT?",
    "retry_count":0,})
print(result['answer'])

app.get_graph().print_ascii()

Based on the provided context, ChatGPT can be used in the following areas:

1. Education field: ChatGPT is commonly used for question and answer testing in the education sector. Students and learners can use ChatGPT to learn, compare, and verify answers for different academic subjects such as physics, mathematics, and chemistry.

2. Introductory tasks: ChatGPT offers a responsive welcome program that maintains attackers' interest in multiple queries. This suggests it can be used for introductory or initial interactions.

3. Code generation: The context mentions challenges with ChatGPT in the field of code generation. While it has limitations, it can potentially be used to assist with coding tasks, particularly in programming languages like Python, C++, and Java.

However, the context also notes that ChatGPT's application scope is limited in code generation due to its training data being biased towards certain programming languages. So while it can be used for coding assistance, it may 

#### 검색 품질 개선
- Multi-Qurey : 여러 관점의 query로 검색 (모호한 질문, 넓은 검색 범위 필요)
- HyDE : 가상 문서를 생성하여 검색(query와 문서의 표현차이가 큰경우)
- Self-RAG : 답변을 평가하고 반복 개선

In [21]:
class MultiQueryState(TypedDict):
    query:str
    sub_queries:List[str]
    retrieved_docs:List[Document]
    answer:str

def generate_sub_queries(state):
    """원본 질문을 여러 관점의 하위 쿼리로 분해"""
    prompt = """\
다음 질문에 대해 서로 다른 관점의검색 쿼리 3개를 생성하세요.
각 쿼리를 줄바꿈으로 구분하세요

원본 질문:
{query}
"""    
    response = watson_llm.invoke(prompt.format(query=state['query']))
    # sub_query
    sub_queries = [q for q in response.content.strip().split('\n') if q.strip()]
    print(f"sub queries {sub_queries}")

    return {"sub_queries":sub_queries}

def multi_retrieve(state):
    """각 하위 쿼리로 검색하고 결과 합치기"""

    all_docs = []
    seen_contents = set()
    for sub_query in state['sub_queries']:
        docs = vectorstore.similarity_search(sub_query, k=3)
        for doc in docs:
            if doc.page_content not in seen_contents:
                all_docs.append(doc)
                seen_contents.add(doc.page_content)

    return {"retrieved_docs":all_docs}

def generate(state):

    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    
    prompt = """\
다음 컨텍스트를 참고하여 질문에 답하세요.
컨텍스트에 없는 내용은 모른다고 답하세요.

컨텍스트:
{context}
질문:
{query}
"""

    response = watson_llm.invoke(prompt.format(context=context,query=state["query"]))
    return {"answer":response.content}   

graph = StateGraph(MultiQueryState)
graph.add_node("generate_sub_queries",generate_sub_queries)
graph.add_node("multi_retrieve",multi_retrieve)
graph.add_node("generate",generate)

graph.add_edge(START,"generate_sub_queries")
graph.add_edge("generate_sub_queries","multi_retrieve")
graph.add_edge("multi_retrieve","generate")
graph.add_edge("generate",END)

app = graph.compile()
result = app.invoke({
    "query":"where can i use chatGPT?"})
print(result['answer'])

app.get_graph().print_ascii()

sub queries ['1. 어디서 ChatGPT를 사용할 수 있는지 알아보고 싶어요. ', '2. ChatGPT를 활용할 수 있는 다양한 플랫폼이나 서비스에 대해 알고 싶어요. ', '3. ChatGPT를 사용할 수 있는 다양한 상황이나 환경에 대해 조사하고 싶어요.']
제공된 컨텍스트에 따르면, ChatGPT는 다음과 같은 분야에서 사용될 수 있습니다:

1. 교육 분야: ChatGPT는 교육 분야에서 질문과 답변 테스트에 일반적으로 사용됩니다. 사용자는 물리학, 수학, 화학 등 다양한 학문 분야에서 학습, 비교 및 답변 검증을 위해 ChatGPT를 사용할 수 있습니다.

2. 데이터 시각화, 정보 추출, 데이터 강화, 품질 평가 및 멀티모달 데이터 처리: ChatGPT는 현재 다양한 응용 분야에서 사용되고 있으며, 이러한 분야에서도 활용될 수 있습니다.

컨텍스트에는 ChatGPT의 사용에 대한 추가적인 정보가 없으므로, 이 외의 분야에서의 사용에 대해서는 알려드리지 못합니다.
      +-----------+      
      | __start__ |      
      +-----------+      
            *            
            *            
            *            
+----------------------+ 
| generate_sub_queries | 
+----------------------+ 
            *            
            *            
            *            
   +----------------+    
   | multi_retrieve |    
   +----------------+    
            *            
            *            
            *            
      +----------+       
    

In [24]:
class HyDEState(TypedDict):
    query:str
    hypothetical_doc:str
    retrieved_docs:List[Document]
    answer:str


def generate_hypothetical(state):
    """질문에 대한 가상의 답변 문서를 생성합니다"""
    
    prompt = """\
다음 질문에 대한 답변이 될 만한 문서를 작성하세요.
실제 정확한 답변이 아니어도 됩니다. 관련 용어와 개념을 포함하면 됩니다.
반드시 2~3문장 이내, 공백 포함 300자(또는 약 100단어) 이하로 짧고 핵심만 작성하세요.

질문:
{query}
"""

    response = watson_llm.invoke(prompt.format(query=state["query"]))
    return {"hypothetical_doc":response.content}

def hyde_retrieve(state):
    """가상 문서를 쿼리로 사용하여 검색"""
    docs = vectorstore.similarity_search(state['hypothetical_doc'], k=3)

    return {"retrieved_docs":docs}

def generate(state):

    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    
    prompt = """\
다음 컨텍스트를 참고하여 질문에 답하세요.
컨텍스트에 없는 내용은 모른다고 답하세요.

컨텍스트:
{context}
질문:
{query}
"""

    response = watson_llm.invoke(prompt.format(context=context,query=state["query"]))
    return {"answer":response.content}

graph = StateGraph(HyDEState)
graph.add_node("generate_hypothetical",generate_hypothetical)
graph.add_node("hyde_retrieve",hyde_retrieve)
graph.add_node("generate",generate)

graph.add_edge(START,"generate_hypothetical")
graph.add_edge("generate_hypothetical","hyde_retrieve")
graph.add_edge("hyde_retrieve","generate")
graph.add_edge("generate",END)

app = graph.compile()
result = app.invoke({
    "query":"where can i use chatGPT?"})
print(result['answer'])

app.get_graph().print_ascii()

제공된 컨텍스트에 따르면, ChatGPT는 교육 분야에서 질문과 답변 테스팅에 일반적으로 사용됩니다. 사용자는 ChatGPT를 이용하여 물리학, 수학, 화학 등 다양한 학문 분야의 다른 학문적 주제에 대한 답변을 학습, 비교 및 검증할 수 있습니다. 또한, Treude et al. [39]는 ChatGPT를 "GPTCOM-CARE" 프로토타입에 통합하여 프로그래밍 질문 문제를 해결했으며, 이를 통해 동일한 질문에 대해 여러 소스 코드 솔루션을 생성할 수 있었습니다. 이러한 통합은 비기술 사용자가 시스템과 상호 작용하는 것을 더 쉽게 만들어, 전문 지식이나 교육의 필요성을 줄여줍니다.
      +-----------+        
      | __start__ |        
      +-----------+        
            *              
            *              
            *              
+-----------------------+  
| generate_hypothetical |  
+-----------------------+  
            *              
            *              
            *              
    +---------------+      
    | hyde_retrieve |      
    +---------------+      
            *              
            *              
            *              
      +----------+         
      | generate |         
      +----------+         
            *              
            *              
            *             

In [28]:
# self refind rag
# 답변을 생성 => 평가 => 재검색 or 답변 수정
class SelfRefindRAGState(TypedDict):
    query:str
    retrieved_docs:List[Document]
    answer:str
    evaluation:str
    retry_count:int

def retrieve(state):
    # 기존 벡터스토어 질의
    vectorstore = Chroma(collection_name="research", embedding_function=watsonx_enbedding,persist_directory="./db/chroma_db")

    docs = vectorstore.similarity_search(state['query'], k=3)

    return {"retrieved_docs":docs}

def generate(state):

    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    
    prompt = """\
다음 컨텍스트를 참고하여 질문에 답하세요.
컨텍스트에 없는 내용은 모른다고 답하세요.
컨텍스트에 정보가 부족하면 그 사실을 명시하세요.

컨텍스트:
{context}

질문:
{query}
"""

    response = watson_llm.invoke(prompt.format(context=context,query=state["query"]))
    return {"answer":response.content}

def evaluate(state):
    """생성한 답변이 답변의 충실도 관련성을 평가"""
    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    prompt = ChatPromptTemplate.from_template(
        """
질문:
{query}

컨텍스트:
{context}

답변:
{answer}

반드시 아래 줄 중 하나로만 답하세요. 
'sufficient'
'insufficient'
"""
    )
    response = watson_llm.invoke(prompt.format(query=state['query'],answer=state['answer'],context=context))
    content = response.content.lower().strip()
    evaluation = "insufficient" if content.startswith("insufficient") else "sufficient"

    return {"evaluation":evaluation}

def refine(state):
    """평가 결과를 반영하여 답변을 개선합니다."""
    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    
    prompt = """\
다음 답변을 개선하세요
원래 질문:
{query}

컨텍스트:
{context}
이전답변:
{answer}
컨텍스트에 더 출실하고 질문에 더 정확히 답하도록 수정하세요.
"""

    response = watson_llm.invoke(prompt.format(query=state['query'],answer=state['answer'],context=context))
    return {"answer":response.content, "retry_count":state["retry_count"]+1}

# route
def route_after_eval(state):
    """재검사 여부 결정"""
    if state['evaluation']=="sufficient" or state['retry_count'] >=2:
        return "done"
    return "refine"

# 그래프 구성
graph = StateGraph(SelfRefindRAGState)
graph.add_node("retrieve",retrieve)
graph.add_node("generate",generate)
graph.add_node("evaluate",evaluate)
graph.add_node("refine",refine)

graph.add_edge(START,"retrieve")
graph.add_edge("retrieve","generate")
graph.add_edge("generate","evaluate")

graph.add_conditional_edges("evaluate",route_after_eval,{"refine": "refine","done":END})
graph.add_edge("refine","evaluate")

app = graph.compile()
result = app.invoke({
    "query":"where can i use chatGPT?",
    "retry_count":0,})
print(result['answer'])

app.get_graph().print_ascii()

Based on the provided context, ChatGPT can be utilized in the following areas:

1. Education field: ChatGPT is commonly used for question and answer testing in the education sector. Students and users can leverage ChatGPT to learn, compare, and verify answers for various academic subjects such as physics, mathematics, and chemistry. This application can assist in enhancing the learning experience and providing additional support for students.

2. Code generation: The context highlights that ChatGPT has applications in the field of code generation. However, it also points out that there are several challenges associated with using ChatGPT for code generation. One of the main challenges is the limited application scope due to the training data being biased towards programming languages like Python, C++, and Java. This bias may make ChatGPT less suitable for certain programming languages or coding styles.

3. Introductory tasks and responsive welcome program: The context mentions that Cha

In [30]:
# self rag
# 답변을 생성 => 평가 => 부족 => 질문 개선 => 검색 => 새문서 생성
class SelfRAGState(TypedDict):
    query:str
    retrieved_docs:List[Document]
    answer:str
    evaluation:str
    retry_count:int

def retrieve(state):
    # 기존 벡터스토어 질의
    vectorstore = Chroma(collection_name="research", embedding_function=watsonx_enbedding,persist_directory="./db/chroma_db")

    docs = vectorstore.similarity_search(state['query'], k=3)

    return {"retrieved_docs":docs}

def generate(state):

    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    
    prompt = """\
다음 컨텍스트를 참고하여 질문에 답하세요.
컨텍스트에 없는 내용은 모른다고 답하세요.
컨텍스트에 정보가 부족하면 그 사실을 명시하세요.

컨텍스트:
{context}

질문:
{query}
"""

    response = watson_llm.invoke(prompt.format(context=context,query=state["query"]))
    return {"answer":response.content}

def evaluate(state):
    """생성한 답변이 답변의 충실도 관련성을 평가"""
    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    prompt = ChatPromptTemplate.from_template(
        """
질문:
{query}

컨텍스트:
{context}

답변:
{answer}

반드시 아래 줄 중 하나로만 답하세요. 
'sufficient'
'insufficient'
"""
    )
    response = watson_llm.invoke(prompt.format(query=state['query'],answer=state['answer'],context=context))
    content = response.content.lower().strip()
    evaluation = "insufficient" if content.startswith("insufficient") else "sufficient"

    return {"evaluation":evaluation}

def rewrite(state):
    """평가 결과를 반영하여 질문을 개선합니다."""
    context = "\n\n".join(doc.page_content for doc in state['retrieved_docs'])
    
    prompt = """\
원래 질문:
{query}
검색 결과가 충분하지 않습니다.
더 구체적이고 검색하기 좋은 질문으로 재작성하세요.
질문만 출력하세요.
"""
    response = watson_llm.invoke(prompt.format(query=state['query']))
    return {"answer":response.content, "retry_count":state["retry_count"]+1}

# route
def route_after_eval(state):
    """재검사 여부 결정"""
    if state['evaluation']=="sufficient" or state['retry_count'] >=2:
        return "done"
    return "retry"

# 그래프 구성
graph = StateGraph(SelfRAGState)
graph.add_node("retrieve",retrieve)
graph.add_node("generate",generate)
graph.add_node("evaluate",evaluate)
graph.add_node("rewrite",rewrite)

graph.add_edge(START,"retrieve")
graph.add_edge("retrieve","generate")
graph.add_edge("generate","evaluate")

graph.add_conditional_edges("evaluate",route_after_eval,{"retry": "rewrite","done":END})
graph.add_edge("rewrite","retrieve")

app = graph.compile()
result = app.invoke({
    "query":"where can i use chatGPT?",
    "retry_count":0,})
print(result['answer'])

app.get_graph().print_ascii()

Based on the provided context, ChatGPT can be used in the following ways:

1. Education field: ChatGPT is commonly used for question and answer testing in the education sector. Students and learners can use ChatGPT to learn, compare, and verify answers for different academic subjects such as physics, mathematics, and chemistry.

2. Introductory tasks: ChatGPT offers a responsive welcome program that maintains attackers' interest in multiple queries. This suggests that it can be used for introductory or initial interactions.

3. Code generation: The context mentions that ChatGPT has some limitations in the field of code generation. It is biased towards programming languages like Python, C++, and Java, which may make it unsuitable for some programming languages or coding styles. However, it is still used for code generation tasks.

The context does not provide any other specific information about where ChatGPT can be used. It mainly focuses on its applications in the education field, int

### LLM 애플리케이션 성능 최적화 전략
- 1. 비용 절감
    - InMemoryCache,SQLiteCache 모델 경량화
- 2. 응답속도 향상
    - RunnableParallel, batch, Streaming
- 3. 처리량 향상
    - ainvoke(),asyncio.gather()

In [11]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache, SQLiteCache
import time

In [ ]:
text = "파이썬이란 무엇인가요?"

print("===============캐쉬 없음===============")
set_llm_cache(None)

start =time.time()
r1 = watson_llm.invoke(text)
print(f"1회차 : {time.time()-start:.2f}초")
start =time.time()
r1 = watson_llm.invoke(text)
print(f"2회차 : {time.time()-start:.2f}초")

# InMemoryCache
print("===============InMemoryCache===============")
set_llm_cache(InMemoryCache())
start =time.time()
r2 = watson_llm.invoke(text)
print(f"1회차 : {time.time()-start:.2f}초")
start =time.time()
r2 = watson_llm.invoke(text)
print(f"2회차 : {time.time()-start:.2f}초")

# SQLiteCache
print("===============SQLiteCache===============")
set_llm_cache(SQLiteCache(database_path="./db/llm_cache.db"))
start =time.time()
r3 = watson_llm.invoke(text)
print(f"1회차 : {time.time()-start:.2f}초")
start =time.time()
r3 = watson_llm.invoke(text)
print(f"2회차 : {time.time()-start:.2f}초")

===============캐쉬 없음===============
1회차 : 4.58초
2회차 : 3.33초
===============InMemoryCache===============
1회차 : 2.79초
2회차 : 0.01초
===============SQLiteCache===============
1회차 : 2.79초
2회차 : 0.00초


c:\souce\ollama\.venv\Lib\site-packages\langchain_community\cache.py:265: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(row[0]) for row in rows]
